# 02 · Your First Tool Call

A model can only produce text. It cannot check the weather, open a file or call an API. A **tool** is a normal Python function that the model can *ask you* to run.

The most important idea in this module:

> **The model never runs the function. It only asks. Your code runs it and hands the result back.**

```
You ask a question ──> Model says "please call get_weather('Pune')"
                                      │
Your code runs get_weather ───────────┘
        │
        └──> You send the result back ──> Model writes the final answer
```

That means **two API calls** for one question. We will do each step by hand.

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("✅ Ready")

✅ Ready


## Step 1: write the function
For now it returns fake data, so we can focus on how tool calling works. In the next notebook we use a real weather API.

In [3]:
def get_weather(city):
    """A pretend weather lookup."""
    return f"It is 24°C and sunny in {city}."

print(get_weather("Pune"))

It is 24°C and sunny in Pune.


## Step 2: describe the tool to the model
The model cannot see our Python code. We give it a short description: the **name**, what it **does**, and what **inputs** it needs.
The model reads the description to decide when the tool is useful.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather (temperature and rain) for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Pune"},
                },
                "required": ["city"],
            },
        },
    },
]

## Step 3: ask a question and pass the tools
Notice `tools=tools`. That tells the model which tools it is allowed to ask for.

In [5]:
messages = [{"role": "user", "content": "What is the weather in Pune?"}]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print("Text reply :", message.content)   # None: the model did not answer, it asked for a tool

Text reply : None


## Step 4: read the model's request
The model did not write an answer. It returned a **tool call**: the tool name and the inputs (`arguments`, written as JSON text).

In [6]:
tool_call = message.tool_calls[0]

print("Tool wanted :", tool_call.function.name)
print("Arguments   :", tool_call.function.arguments)

args = json.loads(tool_call.function.arguments)   # turn the JSON text into a Python dict
print(args)

Tool wanted : get_weather
Arguments   : {"city":"Pune"}
{'city': 'Pune'}


## Step 5: run the function ourselves
`**args` means "use the dict's keys as the function's inputs", so `get_weather(**{"city": "Pune"})` is `get_weather(city="Pune")`.

In [7]:
result = get_weather(**args)
print(result)

It is 24°C and sunny in Pune.


## Step 6: send the result back
We add two things to the conversation:
1. the model's own request (so it remembers what it asked for), and
2. a message with the role **`tool`** that holds our result. This is the `tool` role from Module 01.

Then we call the model again.

In [8]:
messages.append(message)    # the model's request

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,    # says which request this result belongs to
    "content": result,
})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print(final.choices[0].message.content)

The weather in Pune is currently 24°C and sunny.


## The model decides when to use a tool
Tools are optional. If a question does not need one, the model just answers.

In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
    tools=tools,
)
message = response.choices[0].message

print("Tool calls :", message.tool_calls)   # None: no tool needed
print("Answer     :", message.content)

Tool calls : None
Answer     : 2 + 2 equals 4.


## Recap
* A tool is a normal function plus a short description for the model.
* The model **asks**, your code **runs**, and you send the result back with the `tool` role.
* One question with a tool takes **two** model calls.

Next: replace the fake weather with a real API.